# EDA — Transfermarkt Player Market Values
**Source**: `transfermarkt/` — mexwell Kaggle dataset (CC0): `players.csv`, `player_valuations.csv`, `clubs.csv`, `games.csv`, `appearances.csv`, `game_events.csv`, `club_games.csv`, `competitions.csv`

**Purpose**: Profile player market values as a team talent-depth feature. Check coverage for WC 2026 nations, market value distributions, recency of valuations, and feasibility of squad-level aggregation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA = Path('../data/raw/transfermarkt')

## 1. Load & Basic Profile

In [ ]:
players = pd.read_csv(DATA / 'players.csv', parse_dates=['date_of_birth', 'contract_expiration_date'])
valuations = pd.read_csv(DATA / 'player_valuations.csv', parse_dates=['date'])
clubs = pd.read_csv(DATA / 'clubs.csv')

print('Players shape:', players.shape)
print('Columns:', players.columns.tolist())
print()
print('Valuations shape:', valuations.shape)
print('Columns:', valuations.columns.tolist())
print('Valuation date range:', valuations['date'].min(), '->', valuations['date'].max())
print()
print('Clubs shape:', clubs.shape)
players.head(3)

In [ ]:
print('Missing values in players.csv:')
missing = players.isnull().mean().mul(100).round(1)
print(missing[missing > 0].sort_values(ascending=False))

## 2. Current Market Value Distribution

In [ ]:
players_with_value = players.dropna(subset=['market_value_in_eur'])
print(f'Players with market value: {len(players_with_value)} / {len(players)} ({len(players_with_value)/len(players)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw distribution
players_with_value['market_value_in_eur'].clip(0, 1e8).plot(
    kind='hist', bins=60, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Current Market Value Distribution (capped at €100M)')
axes[0].set_xlabel('Market Value (€)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x/1e6:.0f}M'))

# Log distribution
np.log10(players_with_value['market_value_in_eur'].clip(1)).plot(
    kind='hist', bins=40, ax=axes[1], color='coral', edgecolor='white'
)
axes[1].set_title('Market Value Distribution (log10 scale)')
axes[1].set_xlabel('log10(Market Value in €)')
axes[1].set_xticks([4, 5, 6, 7, 8])
axes[1].set_xticklabels(['10K', '100K', '1M', '10M', '100M'])

plt.tight_layout()
plt.show()

print(f'Median market value: €{players_with_value["market_value_in_eur"].median()/1e6:.2f}M')
print(f'Mean market value: €{players_with_value["market_value_in_eur"].mean()/1e6:.2f}M')
print(f'Max market value: €{players_with_value["market_value_in_eur"].max()/1e6:.0f}M')
print()
print('Top 10 most valuable players:')
print(players_with_value.nlargest(10, 'market_value_in_eur')[
    ['name', 'country_of_citizenship', 'position', 'market_value_in_eur']
].assign(value_m=lambda x: (x['market_value_in_eur']/1e6).round(1)).drop(columns='market_value_in_eur').to_string(index=False))

## 3. Market Value by Position

In [ ]:
pos_value = players_with_value.groupby('position')['market_value_in_eur'].agg(['median', 'mean', 'count'])
pos_value['median_m'] = pos_value['median'] / 1e6
pos_value['mean_m'] = pos_value['mean'] / 1e6
pos_value = pos_value.sort_values('median', ascending=False)
print('Market value by position:')
print(pos_value[['count', 'median_m', 'mean_m']].round(2))

fig, ax = plt.subplots(figsize=(9, 4))
pos_value['median_m'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Median Market Value by Position')
ax.set_xlabel('Median Value (€M)')
plt.tight_layout()
plt.show()

## 4. Coverage: WC 2026 National Teams

In [ ]:
# WC 2026 qualified teams (48 teams)
wc2026_teams = [
    'Argentina', 'Brazil', 'France', 'England', 'Spain', 'Germany', 'Portugal', 'Netherlands',
    'Belgium', 'Italy', 'Croatia', 'Switzerland', 'Denmark', 'Mexico', 'United States', 'Canada',
    'Uruguay', 'Colombia', 'Ecuador', 'Chile', 'Paraguay', 'Venezuela', 'Peru', 'Bolivia',
    'Morocco', 'Senegal', 'Nigeria', 'Egypt', 'South Africa', 'Cameroon', 'Ivory Coast', 'Ghana',
    'Algeria', 'Tunisia', 'DR Congo', 'Saudi Arabia', 'Japan', 'South Korea', 'Iran', 'Australia',
    'Qatar', 'New Zealand', 'Costa Rica', 'Honduras', 'Panama', 'Jamaica', 'Trinidad and Tobago',
    'Turkey'
]

# Count players per nationality with market values
nat_coverage = players_with_value.groupby('country_of_citizenship').agg(
    n_players=('player_id', 'count'),
    total_value_m=('market_value_in_eur', lambda x: x.sum() / 1e6),
    median_value_m=('market_value_in_eur', lambda x: x.median() / 1e6),
    top18_value_m=('market_value_in_eur', lambda x: x.nlargest(18).sum() / 1e6)
).reset_index()

# Filter to WC teams
wc_coverage = nat_coverage[nat_coverage['country_of_citizenship'].isin(wc2026_teams)]
missing_teams = [t for t in wc2026_teams if t not in nat_coverage['country_of_citizenship'].values]

print(f'WC 2026 teams with coverage: {len(wc_coverage)} / {len(wc2026_teams)}')
if missing_teams:
    print(f'Missing teams (may need name harmonization): {missing_teams}')

print('\nTop 20 WC nations by top-18 squad value:')
print(wc_coverage.nlargest(20, 'top18_value_m')[
    ['country_of_citizenship', 'n_players', 'top18_value_m', 'median_value_m']
].round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
plot_data = wc_coverage.sort_values('top18_value_m')
ax.barh(plot_data['country_of_citizenship'], plot_data['top18_value_m'], color='steelblue')
ax.set_title('WC 2026 Nations — Top-18 Players Total Market Value (€M)')
ax.set_xlabel('Top-18 Market Value (€M)')
plt.tight_layout()
plt.show()

## 5. Valuation Freshness

In [ ]:
valuations_sorted = valuations.sort_values('date')
print('Valuation history shape:', valuations.shape)
print('Date range:', valuations['date'].min(), '->', valuations['date'].max())

# How many players have valuations in the last year (2025+)?
recent_valuations = valuations[valuations['date'] >= '2025-01-01']
print(f'\nValuations from 2025+: {len(recent_valuations)} records for {recent_valuations["player_id"].nunique()} players')

fig, ax = plt.subplots(figsize=(14, 4))
valuations.set_index('date')['market_value_in_eur'].resample('QE').count().plot(ax=ax)
ax.set_title('Number of Valuation Records per Quarter')
ax.set_xlabel('Quarter')
ax.set_ylabel('# Records')
plt.tight_layout()
plt.show()

## 6. Market Value Skew & Log-Normality

In [ ]:
from scipy import stats

vals = players_with_value['market_value_in_eur'].clip(1)
log_vals = np.log(vals)

print(f'Skewness (raw): {vals.skew():.2f}  (>1 = right-skewed, consider log transform for modeling)')
print(f'Skewness (log): {log_vals.skew():.2f}')
print(f'Kurtosis (raw): {vals.kurtosis():.2f}')
print(f'Kurtosis (log): {log_vals.kurtosis():.2f}')

# QQ plot for log values
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
stats.probplot(log_vals.sample(min(5000, len(log_vals)), random_state=42), dist='norm', plot=axes[0])
axes[0].set_title('QQ Plot — log(Market Value)')

axes[1].scatter(log_vals.sample(min(2000, len(log_vals)), random_state=42),
                np.random.normal(0, 1, min(2000, len(log_vals))), alpha=0.1, s=5)
axes[1].set_xlabel('log(Market Value)')
axes[1].set_title('log(MV) sample vs Normal noise')

plt.tight_layout()
plt.show()
print('Recommendation: use log(market_value) as squad feature input to avoid scale dominance by top players')

## 7. Red Flags Summary

In [ ]:
import datetime
red_flags = []
today = datetime.date(2026, 6, 12)

missing_pct = players['market_value_in_eur'].isnull().mean() * 100
if missing_pct > 30:
    red_flags.append(f'{missing_pct:.1f}% of players have no market value — squad aggregate will drop players; check if WC squad players are covered')

latest_val_date = valuations['date'].max()
days_stale = (today - latest_val_date.date()).days
if days_stale > 90:
    red_flags.append(f'Latest valuation is {days_stale} days old ({latest_val_date.date()}) — misses June 2026 squad fitness; acceptable for pre-tournament squad estimation')

if missing_teams:
    red_flags.append(f'WC 2026 team name mismatches needing harmonization: {missing_teams[:5]}')

if red_flags:
    print('RED FLAGS:')
    for f in red_flags:
        print(' -', f)
else:
    print('No red flags.')

print()
print('MODELING NOTE: Use log(sum of top-18 market values) as squad strength feature.')
print('Need to join players to national teams via squad lists (see notebook 06).')